# CyberLoRA：LoRA 一键训练 + InstantID 免训练出图（Colab T4 16GB）

- **Part 1（本页上方）**：把 20~30 张本人照片训练成专属人物 LoRA（`cyberboy` 触发词）。
- **Part 2（本页底部）**：InstantID 免训练出图 —— 一张照片即可出图，约 3 分钟，无需训练。


In [ ]:
# Step 0：环境检查（确认 T4 GPU 与显存）
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print('CUDA:', torch.cuda.is_available(), '| 设备:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无')
assert torch.cuda.is_available(), '请先在 菜单-运行时-更改运行时类型 选择 T4 GPU'

In [ ]:
# Step 1：克隆 kohya-ss/sd-scripts（tag v0.8.7）并安装依赖
# 注意：直接克隆 sd-scripts，不要用 bmaltais/kohya_ss（子模块无克隆为空）
# 若与 Colab 预装 torch 冲突，去掉 -b v0.8.7 用最新版
!git clone --depth 1 -b v0.8.7 https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q --upgrade accelerate transformers ftfy tensorboard safetensors albumentations opencv-python

In [ ]:
# Step 2：上传训练集
# 左侧文件面板把 100_cyberboy 文件夹拖到 /content/sd-scripts/train_data/
# 目录结构：train_data/100_cyberboy/*.png + 同名 .txt（100 是 kohya 的 repeats 前缀）
import glob, os
imgs = sorted(glob.glob('train_data/100_cyberboy/*.png'))
print(f'训练图：{len(imgs)} 张')
print('示例：', imgs[:3])
assert imgs, '尚未上传：请把 100_cyberboy 文件夹上传到 /content/sd-scripts/train_data/'

In [ ]:
# Step 3：wd14-tagger 自动打标（自动探测 ONNX 输入名 + CPU 回退）
import glob, os, subprocess

def tag_dir(img_dir):
    imgs = sorted(glob.glob(os.path.join(img_dir, '*.png')))
    imgs += sorted(glob.glob(os.path.join(img_dir, '*.jpg')))
    assert imgs, f'{img_dir} 下没有图片，请先完成 Step 2'
    # 自动探测 ONNX 输入名（不同 onnxruntime 版本 input name 不同，硬编码会 KeyError）
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession('wd14_tagger_model.onnx', providers=['CPUExecutionProvider'])
        print('ONNX 输入名探测:', sess.get_inputs()[0].name)
        del sess
    except Exception:
        print('ONNX 探测失败，走 CPU 回退（脚本默认 provider）')
    cmd = ['python', 'finetune/tag_images_by_wd14_tagger.py',
           '--batch_size', '4',
           '--repo_id', 'SmilingWolf/wd-v1-4-convnext-tagger-v2',
           '--thresh', '0.35', img_dir]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)

tag_dir('train_data/100_cyberboy')
print('打标完成：每张图旁生成同名 .txt（触发词 cyberboy 由 keep_tokens=1 保留）')

In [ ]:
# Step 4：单卡非交互式 accelerate 配置（必做，否则 launch 卡在交互问答）
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='fp16')
print('accelerate 配置完成（单卡 fp16）')

In [ ]:
# Step 5：训练脚本（f-string 生成）并启动
# T4 16GB 显存压缩组合：fp16 + AdamW8bit + gradient_checkpointing + cache_latents_to_disk + dim32/alpha16
# OOM 时：network_dim 降到 16 或关 cache_latents；确保 train_batch_size=1
train_script = f'''import subprocess

cmd = [
    "python", "-m", "accelerate.commands.launch",
    "--num_processes=1",
    "--num_machines=1",
    "--mixed_precision=fp16",
    "sdxl_train_network.py",
    "--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0",
    "--train_data_dir=./train_data",
    "--output_dir=./outputs",
    "--output_name=cyberboy_sdxl",
    "--resolution=1024,1024",
    "--mixed_precision=fp16",
    "--no_half_vae",
    "--optimizer_type=AdamW8bit",
    "--gradient_checkpointing",
    "--cache_latents",
    "--cache_latents_to_disk",
    "--network_module=networks.lora",
    "--network_dim=32",
    "--network_alpha=16",
    "--enable_bucket",
    "--min_bucket_reso=512",
    "--max_bucket_reso=1536",
    "--noise_offset=0.1",
    "--train_batch_size=1",
    "--max_train_epochs=8",
    "--learning_rate=1e-4",
    "--text_encoder_lr=5e-5",
    "--lr_scheduler=cosine",
    "--seed=42",
    "--keep_tokens=1",
    "--save_every_n_epochs=1",
    "--save_precision=fp16",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)
'''

# 无 CUDA 时的验证方式：正则提取 f-string 生成的训练脚本 → compile() 静态语法校验
compile(train_script, '<train_script>', 'exec')
print('训练脚本静态语法校验通过')

# 启动训练（约 20~45 分钟 / 8 epochs）
exec(compile(train_script, '<train_script>', 'exec'))

In [ ]:
# Step 6：确认产物（.safetensors 0KB 说明训练异常终止，需至少跑完 1 个 epoch）
import glob, os
outs = sorted(glob.glob('outputs/*.safetensors'))
for o in outs:
    print(f'{o}  {os.path.getsize(o)} bytes')
if not outs:
    print('尚未产出 .safetensors：训练还在进行或未启动，回到 Step 5 检查日志')

In [ ]:
# Step 7：下载 LoRA 产物（.safetensors 50~100MB）
import glob, os
from google.colab import files
for o in sorted(glob.glob('outputs/*.safetensors')):
    if os.path.getsize(o) > 0:
        files.download(o)
        print(f'已触发下载：{o}')

# 下载后：
# 本地推理  ./venv-infer/bin/python inference.py -s studio --lora ./cyberboy_sdxl.safetensors --compare --ref 本人照片目录
# WebUI/ComfyUI  把 .safetensors 放进 models/Lora/，套用 prompts_generated.md 的 Prompt（触发词 cyberboy 放最前）

# 路线 1：InstantID 免训练出图（约 3 分钟）

**不用训练**：身份信息来自人脸 embedding（IP-Adapter 注入）+ 关键点约束布局（ControlNet）。
本 Part 与上方 LoRA 训练部分互相独立，可单独从头运行（模型约 11GB，只下载一次）。

流程：装依赖 → 下载模型 → 上传本人照 → 出图 + 相似度评分。

In [ ]:
# 依赖安装（与训练 Part 独立；insightface 必须 0.7.3，1.x 不兼容 antelopev2）
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.32.2", "transformers==4.46.3", "accelerate", "safetensors",
    "insightface==0.7.3", "onnxruntime", "opencv-python", "huggingface_hub"],
    check=True)
print("依赖 OK")

In [ ]:
# 下载模型（一次性，约 11GB；Colab 磁盘 78GB 足够）
import os
from huggingface_hub import snapshot_download

# 1) vendored InstantID 管线（优先用仓库已测试版本，仓库私有则回退 InstantX 官方）
for url in ("https://raw.githubusercontent.com/tainger/CyberLoRA/main/pipeline_stable_diffusion_xl_instantid.py",
            "https://raw.githubusercontent.com/InstantID/InstantID/main/pipeline_stable_diffusion_xl_instantid.py"):
    if os.system(f"wget -q -O pipeline_stable_diffusion_xl_instantid.py {url}") == 0 \
       and os.path.getsize("pipeline_stable_diffusion_xl_instantid.py") > 10000:
        break
assert os.path.getsize("pipeline_stable_diffusion_xl_instantid.py") > 10000, "pipeline 下载失败"
# 1b) ip_adapter 包（管线 import 需要；仓库自带，回退 InstantX/IP-Adapter）
os.makedirs("ip_adapter", exist_ok=True)
for f in ("__init__.py", "resampler.py", "utils.py", "attention_processor.py"):
    for base in ("https://raw.githubusercontent.com/tainger/CyberLoRA/main/ip_adapter",
                 "https://raw.githubusercontent.com/InstantX/IP-Adapter/main/ip_adapter"):
        if os.system(f"wget -q -O ip_adapter/{f} {base}/{f}") == 0 and os.path.getsize(f"ip_adapter/{f}") > 100:
            break
assert all(os.path.getsize(f"ip_adapter/{f}") > 100 for f in ("__init__.py", "resampler.py", "utils.py", "attention_processor.py")), "ip_adapter 下载失败"


# 2) SDXL 底模 fp16 变体（约 6.5GB；旧版 huggingface_hub 的
#    snapshot_download 不支持 variant 参数，改用 allow_patterns 只拉 fp16 文件）
snapshot_download("stabilityai/stable-diffusion-xl-base-1.0",
                  local_dir="models/sdxl",
                  allow_patterns=["*.fp16.safetensors", "*.json", "*.txt"])

# 3) InstantID：ip-adapter.bin（身份注入）+ ControlNetModel/（关键点约束），约 3.9GB
snapshot_download("InstantX/InstantID", local_dir="models/InstantID")

# 4) antelopev2 人脸分析（检测 + 512 维身份 embedding），约 400MB
snapshot_download("DIAMONIK7777/antelopev2", local_dir="models/antelopev2")
print("模型下载完成")

In [ ]:
# 上传参考照（正脸、清晰、单人；与本地 assets/avatar.jpg 相同要求）
from google.colab import files
up = files.upload()
ref_path = next(iter(up))
print("参考照：", ref_path)

In [ ]:
# 人脸分析：resize 1024 → 取最大脸 → 512 维身份 embedding + 关键点条件图
# （与仓库 instantid_infer.py 完全相同的逻辑）
import math
import cv2, numpy as np
from PIL import Image
from insightface.app import FaceAnalysis

app = FaceAnalysis(name="antelopev2", root=".")  # 读 ./models/antelopev2
app.prepare(ctx_id=0, det_size=(640, 640))

def largest_face(bgr):
    faces = app.get(bgr)
    if not faces:
        return None
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))

def draw_kps(kps, size=640):
    stickwidth = 4
    limb_seq = np.array([[0, 2], [1, 2], [3, 2], [4, 2]])
    color_list = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]
    out = np.zeros([size, size, 3], dtype=np.uint8)
    for i in range(len(limb_seq)):
        idx = limb_seq[i]
        color = color_list[idx[0]]
        x = kps[idx][:, 0]; y = kps[idx][:, 1]
        length = ((x[0]-x[1])**2 + (y[0]-y[1])**2) ** 0.5
        angle = math.degrees(math.atan2(y[0]-y[1], x[0]-x[1]))
        poly = cv2.ellipse2Poly((int(np.mean(x)), int(np.mean(y))),
                                (int(length/2), stickwidth), int(angle), 0, 360, 1)
        out = cv2.fillConvexPoly(out.copy(), poly, color)
    out = (out * 0.6).astype(np.uint8)
    for i, kp in enumerate(kps):
        out = cv2.circle(out.copy(), (int(kp[0]), int(kp[1])), 10, color_list[i], -1)
    return out

bgr = cv2.resize(cv2.imread(ref_path), (1024, 1024))
face = largest_face(bgr)
assert face is not None, "参考照未检测到人脸，请换一张正脸清晰单人照"
face_emb = face["embedding"]            # (512,) 身份向量
face_kps = Image.fromarray(draw_kps(face["kps"] * (640 / 1024.0)))
print(f"人脸提取完成 det_score={face['det_score']:.3f}")

In [ ]:
# 场景库（与仓库 prompt_gen.py SCENES 一致；InstantID 路线不需要触发词）
SCENES = {
    "business":  ("商务正装", "1boy, wearing black suit, white shirt, necktie, office background, professional, corporate headshot, natural light, 8k"),
    "studio":    ("棚拍肖像", "1boy, portrait, studio lighting, bokeh, detailed face, looking at camera, professional photography, high quality, sharp focus"),
    "cyberpunk": ("赛博朋克", "1boy, standing on Tokyo street, neon lights, rain, night, cyberpunk, 8k, cinematic lighting"),
    "astronaut": ("科幻宇航员", "1boy, astronaut in spacesuit, walking on Mars surface, red desert, sci-fi, cinematic, hdr, 8k"),
    "wuxia":     ("古风侠客", "1boy, ancient Chinese warrior, traditional armor, holding sword, temple in background, ink painting style, epic, 8k"),
    "sports":    ("户外运动", "1boy, surfing on ocean waves, sunset sky, dynamic action pose, splashing water, action photography, golden hour, 8k"),
    "cafe":      ("咖啡厅日常", "1boy, sitting in cozy cafe, holding coffee cup, reading book, window with street view, 35mm photography, depth of field, warm lighting, high quality"),
    "soldier":   ("未来战士", "1boy, futuristic soldier, mechanical armor, glowing visor, destroyed city background, sci-fi, hdr, 8k, unreal engine 5 render"),
}
NEG = "worst quality, low quality, blurry, deformed face, bad anatomy, extra fingers"

import torch
from diffusers import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline

controlnet = ControlNetModel.from_pretrained("models/InstantID/ControlNetModel", torch_dtype=torch.float16)
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    "models/sdxl", controlnet=controlnet, torch_dtype=torch.float16,
    use_safetensors=True, variant="fp16")
pipe.load_ip_adapter_instantid("models/InstantID/ip-adapter.bin")
pipe.set_ip_adapter_scale(0.8)
pipe.enable_model_cpu_offload()   # 低显存兜底，T4 也稳定

def generate(key, steps=30, seed=42, size=640):
    name, pos = SCENES[key]
    g = torch.Generator("cpu").manual_seed(seed)
    img = pipe(image=face_kps, image_embeds=face_emb,
               prompt=pos, negative_prompt=NEG,
               controlnet_conditioning_scale=0.8, ip_adapter_scale=0.8,
               num_inference_steps=steps, guidance_scale=5,
               generator=g, width=size, height=size).images[0]
    out = f"instantid_{key}.png"
    img.save(out)
    print(f"[{key}] {name} 已保存 {out}")
    return out

# 出一张商务正装（换场景：把 key 改成 cyberpunk / wuxia / cafe / sports 等；
# 一次性出全部 8 张：for k in SCENES: generate(k)）
out = generate("business")

# ---- 相似度评分：对生成图再做人脸 embedding，与参考照算余弦 ----
gen_bgr = cv2.resize(cv2.cvtColor(np.asarray(Image.open(out).convert("RGB")),
                                  cv2.COLOR_RGB2BGR), (1024, 1024))
gf = largest_face(gen_bgr)
if gf is None:
    print("评分：生成图未检测到人脸（no_face）")
else:
    sim = float(np.dot(face_emb, gf["embedding"]) /
                (np.linalg.norm(face_emb) * np.linalg.norm(gf["embedding"]) + 1e-8))
    band = "高度相似" if sim >= 0.60 else ("相似" if sim >= 0.45 else ("中等" if sim >= 0.30 else "偏低"))
    print(f"相似度：{sim:.4f}（{band}）")

from IPython.display import display
display(Image.open(out))